# Which treatment plants in India meet the 30 mg/L BOD discharge

standard?

A plant level reanalysis of the CSE evaluation of faecal sludge treatment plants and STP co-treatment systems

We used plant data from a 2023 study by the Centre for Science and Environment (CSE) to find out which treatment plants meet the discharge standard of 30 mg/L for biochemical oxygen demand (BOD). The study sampled 69 plants in eight Indian states. Of these plants, 47 are faecal sludge treatment plants (FSTPs) and 22 are sewage treatment plants (STPs) that co-treat faecal sludge. Based on the average discharge BOD in the summary tables of the report, 61 of the 69 plants (88%) meet the standard. All 22 STPs meet it, whatever their technology (sequencing batch reactor, moving bed biofilm reactor, upflow anaerobic sludge blanket, waste stabilisation pond or activated sludge). Among the FSTPs, 39 of 47 meet the standard, including every sampled plant with geotubes, Mizuchi, electrocoagulation or a packaged STP. The eight plants above the limit are FSTPs that use DEWATS (6 of 33 plants) or a moving bed biofilm reactor (2 of 6 plants). The report itself states that six FSTPs fail, which would mean 63 plants meet the standard. The difference is due to two DEWATS plants, Amethi and Dhenkanal, whose values differ between tables of the report.

In [ ]:
library(dplyr)


Attaching package: 'dplyr'

The following objects are masked from 'package:stats':

    filter, lag

The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union

# 1. Introduction

Faecal sludge treatment plants (FSTPs) and sewage treatment plants (STPs) that co-treat faecal sludge discharge their treated effluent into drains, rivers and fields. In a 2023 report, the Centre for Science and Environment (CSE) compared the effluent of 69 such plants in India with the national discharge standards ([Narain et al. 2023](#ref-narain2023)). For biochemical oxygen demand (BOD), the report uses a limit of 30 mg/L, which it attributes to the Central Pollution Control Board (CPCB) and the Ministry of Environment, Forest and Climate Change (MoEF&CC). The report also mentions a stricter limit of 20 mg/L for metro cities.

We used the plant data in the report to answer two questions:

- How many of the evaluated plants meet the 30 mg/L BOD standard?
- Which treatment technologies do the plants that meet the standard use?

# 2. Methods

## 2.1 Data source

All data come from the Phase II report by CSE ([Narain et al. 2023](#ref-narain2023)), which is stored in this repository as `data/1689832445895.pdf`. CSE sampled the plants over two years starting in 2021, in Telangana, Tamil Nadu, Odisha, Rajasthan, Uttar Pradesh, Uttarakhand, Madhya Pradesh and Chhattisgarh. Of the 69 plants, 47 are FSTPs and 22 are STPs with co-treatment of faecal sludge. For each plant, the report gives the average of the samples taken from the treated discharge water. Four more FSTPs in Odisha (Surada, Kashinagar, Hinjilicut and Talcher) had no outlet water at the time of sampling, so the report has no effluent data for them and we left them out.

## 2.2 Data extraction

We copied the average discharge BOD of each plant from two places in the report into the file `data/bod-discharge.csv`:

- Table 9 (FSTPs) and Table 10 (STPs) in the chapter on results and discussion. We call these the summary tables.
- The outlet rows of Tables 1 to 7 in Annexure II, which hold the consolidated data for each state. For Bharwara, which has several outlet rows, we used the final outlet after chlorination.

We then checked every value in the CSV file against the text of the PDF with a script. All 69 rows matched both sources.

## 2.3 Technology groups

We grouped the plants by technology in the same way as Tables 9 and 10 of the report. The abbreviations are:

- DEWATS, decentralised wastewater treatment system (settler, anaerobic baffled reactor or anaerobic filter, and planted gravel filter)
- MBBR, moving bed biofilm reactor
- SBR, sequencing batch reactor
- UASB, upflow anaerobic sludge blanket
- ASP, activated sludge process

The report groups the Modinagar pilot plant with MBBR in Table 9, although Table 5 describes it as a hyper core (lamella) system. We kept the grouping of Table 9.

## 2.4 Compliance rule

A plant meets the standard when its average discharge BOD is 30 mg/L or lower. We used the summary tables for the main count, because the report uses these tables for its national results. We then repeated the count with the values from Annexure II and compared both counts with the number stated in the text of the report.

# 3. Results

## 3.1 Number of plants that meet the standard

Of the 69 plants, 61 (88%) meet the 30 mg/L BOD standard (<a href="#tbl-tech" class="quarto-xref">Table 1</a>). All 22 STPs with co-treatment meet the standard. Among the FSTPs, 39 of 47 (83%) meet it. The other 8 FSTPs have an average discharge BOD between 30.6 and 90 mg/L.

In [ ]:
fmt_range <- function(lo, hi) {
  if_else(lo == hi, sprintf("%.1f", lo), sprintf("%.1f to %.1f", lo, hi))
}

summarise_group <- function(df) {
  summarise(
    df,
    plants = n(),
    meet = sum(meets_summary),
    lo = min(bod_summary_mgl),
    hi = max(bod_summary_mgl),
    .groups = "drop"
  )
}

by_tech <- bod |>
  group_by(plant_type, technology) |>
  summarise_group() |>
  mutate(
    technology = factor(technology, levels = tech_order),
    type_label = as.character(plant_type)
  ) |>
  arrange(plant_type, technology) |>
  mutate(technology = as.character(technology))

totals <- bind_rows(
  bod |> group_by(plant_type) |> summarise_group() |>
    mutate(type_label = paste("All", if_else(plant_type == "FSTP", "FSTPs", "STPs")), technology = ""),
  bod |> summarise_group() |> mutate(type_label = "All plants", technology = "")
)

bind_rows(by_tech, totals) |>
  mutate(
    type_label = if_else(type_label == lag(type_label, default = ""), "", type_label),
    share = pct(meet, plants),
    range = fmt_range(lo, hi)
  ) |>
  select(type_label, technology, plants, meet, share, range) |>
  kable(
    col.names = c(
      "Plant type", "Technology", "Plants", "Meet 30 mg/L",
      "Share (%)", "Discharge BOD (mg/L)"
    ),
    align = "llrrrl"
  )

## 3.2 Technologies that meet the standard

Every sampled plant with one of the following technologies meets the standard (<a href="#tbl-tech" class="quarto-xref">Table 1</a> and <a href="#fig-bod" class="quarto-xref">Figure 1</a>):

- FSTPs with geotubes (3 plants), Mizuchi (3 plants), electrocoagulation (1 plant) or a packaged STP with pyrolysis (1 plant)
- STPs with SBR (11 plants), MBBR (5 plants), UASB (3 plants), a waste stabilisation pond (2 plants) or ASP (1 plant)

The two FSTP technologies with plants above the limit are also the two most common ones. Of the 33 DEWATS plants, 27 meet the standard. Of the 6 FSTPs with MBBR, 4 meet the standard.

In [ ]:
col_ok <- "#2a78d6"
col_fail <- "#d03b3b"
ink <- "#0b0b0b"
ink_2 <- "#52514e"
grid_col <- "#e1e0d9"

plot_df <- bod |>
  add_count(plant_type, technology) |>
  mutate(
    tech_rank = match(technology, tech_order),
    tech_label = paste0(technology, " (", n, ")"),
    panel = paste0(plant_type, " (", ave(n, plant_type, FUN = length), " plants)"),
    status = factor(
      if_else(meets_summary, "30 mg/L or below", "Above 30 mg/L"),
      levels = c("30 mg/L or below", "Above 30 mg/L")
    )
  )

label_levels <- plot_df |>
  distinct(plant_type, tech_rank, tech_label) |>
  arrange(plant_type, desc(tech_rank)) |>
  pull(tech_label)

plot_df <- mutate(plot_df, tech_label = factor(tech_label, levels = label_levels))

# Plants above the limit sit on the row line, with names placed
# alternately below and above the row so that they do not overlap
fail_df <- plot_df |>
  filter(!meets_summary) |>
  arrange(tech_label, bod_summary_mgl) |>
  group_by(tech_label) |>
  mutate(nudge = if_else(row_number() %% 2 == 1, -0.4, 0.4)) |>
  ungroup()

ggplot(plot_df, aes(x = bod_summary_mgl, y = tech_label)) +
  geom_vline(xintercept = limit, linetype = "dashed", colour = ink_2, linewidth = 0.4) +
  geom_point(
    data = filter(plot_df, meets_summary),
    aes(colour = status, shape = status),
    size = 2.3, alpha = 0.85,
    position = position_jitter(width = 0, height = 0.2, seed = 11)
  ) +
  geom_point(
    data = fail_df,
    aes(colour = status, shape = status),
    size = 2.5, alpha = 0.9
  ) +
  geom_text_repel(
    data = fail_df,
    aes(label = plant),
    nudge_y = fail_df$nudge, direction = "x",
    size = 2.9, colour = ink,
    min.segment.length = 0, segment.colour = "#898781", segment.size = 0.3,
    box.padding = 0.2, max.overlaps = Inf, seed = 11
  ) +
  facet_wrap(vars(panel), ncol = 1, scales = "free_y", space = "free_y") +
  scale_colour_manual(values = c("30 mg/L or below" = col_ok, "Above 30 mg/L" = col_fail)) +
  scale_shape_manual(values = c("30 mg/L or below" = 16, "Above 30 mg/L" = 17)) +
  scale_x_continuous(
    limits = c(0, 95), breaks = seq(0, 90, 10),
    expand = expansion(mult = c(0, 0.01))
  ) +
  labs(x = "Average discharge BOD (mg/L)", y = NULL, colour = NULL, shape = NULL) +
  theme_minimal(base_size = 10) +
  theme(
    legend.position = "top",
    legend.justification = "left",
    panel.grid.major.y = element_blank(),
    panel.grid.minor = element_blank(),
    panel.grid.major.x = element_line(colour = grid_col, linewidth = 0.3),
    axis.text = element_text(colour = ink_2),
    axis.title.x = element_text(colour = ink_2, margin = margin(t = 6)),
    strip.text = element_text(hjust = 0, face = "bold", colour = ink, margin = margin(b = 4)),
    panel.spacing.y = unit(10, "pt")
  )

## 3.3 Plants above the standard

<a href="#tbl-fail" class="quarto-xref">Table 2</a> lists the 8 plants above the standard. Six are DEWATS plants in Tamil Nadu, Uttar Pradesh and Odisha, and two are MBBR plants in Tamil Nadu and Uttar Pradesh. Baripada has the highest value at 90 mg/L, three times the limit. Dhenkanal (30.6 mg/L) and Bhadrak (32.1 mg/L) are only just above the limit.

In [ ]:
bod |>
  filter(!meets_summary) |>
  arrange(desc(bod_summary_mgl)) |>
  select(plant, state, technology, bod_summary_mgl, bod_annex_mgl) |>
  kable(
    col.names = c(
      "Plant", "State", "Technology",
      "Summary table (mg/L)", "Annexure II (mg/L)"
    ),
    align = "lllrr"
  )

The report also mentions the stricter limit of 20 mg/L for metro cities. Khajaguda (20.5 mg/L) and Nanakramguda (21.3 mg/L) are STPs in Hyderabad, a metro city, and both are just above the metro limit. Both plants meet the limit of 30 mg/L that we used here.

## 3.4 Counts from different parts of the report

The number of plants that meet the standard depends on which part of the report we use (<a href="#tbl-sens" class="quarto-xref">Table 3</a>). With the summary tables, 61 plants meet the standard. With Annexure II, 62 plants meet it. The text of the report states that only six FSTPs fail, which gives 63 plants.

In [ ]:
tibble(
  source = c(
    "Summary tables (Tables 9 and 10)",
    "Annexure II",
    "Report text (executive summary and page 37)"
  ),
  fstp_ok = c(sum(fstp$meets_summary), sum(fstp$meets_annex), nrow(fstp) - n_fail_report),
  stp_ok = c(sum(stp$meets_summary), sum(stp$meets_annex), nrow(stp))
) |>
  mutate(all_ok = fstp_ok + stp_ok, share = pct(all_ok, n_all)) |>
  kable(
    col.names = c(
      "Source of the values",
      paste0("FSTPs (of ", nrow(fstp), ")"),
      paste0("STPs (of ", nrow(stp), ")"),
      paste0("All plants (of ", n_all, ")"),
      "Share (%)"
    ),
    align = "lrrrr"
  )

The difference between the summary tables and Annexure II is due to two DEWATS plants:

- Dhenkanal has 30.6 mg/L in Table 9, Graph 4 and Table 13, but 14.0 mg/L in the Odisha chapter (Graph 32) and in Annexure II. The plant is above the limit with the first value and below it with the second.
- Amethi has 48.3 mg/L in Table 9, but 30.1 mg/L in Graph 4, in the Uttar Pradesh chapter and in Annexure II. The report notes that Amethi was sampled only once. The plant is above the limit with both values.

The report does not name the six FSTPs that it counts as failing. Its state chapters name seven plants above the limit:

- Periyanaickenpalayam and Shenkottai in Tamil Nadu
- Balasore, Baripada and Bhadrak in Odisha
- Loni and Amethi in Uttar Pradesh

The same seven plants are above the limit in Annexure II. The count of six in the report most likely leaves out Amethi, whose value of 30.1 mg/L is just above the limit, but we could not confirm this from the report.

# 4. Discussion

Most of the plants in the CSE study meet the BOD discharge standard of 30 mg/L. Depending on the table used, 61 to 63 of the 69 plants meet it. All STPs with co-treatment of faecal sludge meet the standard, whatever their technology. All plants above the limit are FSTPs, and all of them use DEWATS or MBBR.

We cannot conclude from the data that one FSTP technology is better than another, for the following reasons:

- The number of plants per technology is small and uneven. DEWATS was sampled at 33 plants, while electrocoagulation and the packaged STP were each sampled at one plant. For the technologies with one to three plants, the counts apply only to the sampled plants and not to the technology in general.
- The report states that most FSTPs worked below their design capacity and received faecal sludge irregularly ([Narain et al. 2023](#ref-narain2023)). Plants with the same technology can therefore perform differently because of how they are run.
- Each value is an average over one to three sampling rounds. A plant can meet the limit on average and still exceed it on single days.

BOD is only one of the discharge standards. The report states that 32 of the 47 FSTPs and 16 of the 22 STPs were above the COD limit of 50 mg/L, and that many plants were above the limit for faecal coliforms ([Narain et al. 2023](#ref-narain2023)). A plant that meets the BOD standard does not necessarily meet the other standards.

Finally, we copied the data by hand from a PDF report. We checked each value against the text of the PDF, but we could not check the values against the laboratory records of CSE. The report gives different values for Dhenkanal and Amethi in different tables, so at least one value for each plant is wrong. For a firm count of compliant plants, CSE would need to confirm the values for Dhenkanal and Amethi.

# References

Narain, Sunita, Vinod Vijayan, Sama Kalyana Chakravarthy, Megha Tyagi, Arvind Singh Senger, and Ashitha Gopinath. 2023. *Evaluation of FSTPs and STP Co-Treatment Systems Across India: An Insight into Technology and Performance*. Phase II study. Centre for Science and Environment.

# Appendix

In [ ]:
bod |>
  mutate(
    plant_type = factor(plant_type),
    tech_rank = match(technology, tech_order),
    meets = if_else(meets_summary, "Yes", "No")
  ) |>
  arrange(plant_type, tech_rank, state, plant) |>
  select(plant, state, plant_type, technology, bod_summary_mgl, bod_annex_mgl, meets) |>
  kable(
    col.names = c(
      "Plant", "State", "Plant type", "Technology",
      "Summary table (mg/L)", "Annexure II (mg/L)", "Meets 30 mg/L"
    ),
    align = "llllrrl"
  )